# Final NEDI x2 Evaluation — All Datasets

Run this notebook top to bottom in Google Colab after committing and pushing the validated NEDI changes. It evaluates only x2: Set5, Set14, BSD100, and Urban100. Results are checkpointed after every image in a new timestamped Google Drive folder.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'
REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')

if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
print(f'Repository ready: {REPO_ROOT}')


In [ ]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.evaluation.data_validation import validate_prepared_dataset
from app.evaluation.experiment import write_results_csv
from app.evaluation.images import pair_image_paths
from app.evaluation.nedi import NEDIEvaluationConfig, evaluate_nedi_image
from app.evaluation.reporting import summarize_results

print('NEDI x2 evaluator imported successfully.')


In [ ]:
from datetime import UTC, datetime
import csv

DATA_ROOT = Path('/content/drive/MyDrive/FYP_SR_Data')
# Leave as None for a new run. To resume after an interruption, replace
# None with the timestamped folder name printed by the earlier run.
RESUME_RUN_ID = None
RUN_ID = RESUME_RUN_ID or datetime.now(UTC).strftime('%Y%m%d_%H%M%S_utc')
RUN_ROOT = DATA_ROOT / 'results' / 'final_nedi' / 'x2_full' / RUN_ID
METRICS_ROOT = RUN_ROOT / 'metrics'
DATASETS = ('Set5', 'Set14', 'BSD100', 'Urban100')

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset root not found: {DATA_ROOT}')

print(f'Full NEDI x2 run: {RUN_ROOT}')
print('Protocol: 3 warm-ups and 10 timed CPU runs per image.')


In [ ]:
validations = {}
for dataset in DATASETS:
    validation = validate_prepared_dataset(dataset, 2, DATA_ROOT)
    validations[dataset] = validation
    print(f'VALID: {dataset} x2 has {validation.image_count} complete HR/LR pairs.')

print('All x2 datasets passed validation.')


In [ ]:
# A checkpoint CSV is rewritten after every completed image. If the runtime
# disconnects, set RESUME_RUN_ID above and rerun this cell to skip completed images.
all_records = []
for dataset in DATASETS:
    validation = validations[dataset]
    config = NEDIEvaluationConfig(
        dataset=dataset, scale=2, window_size=8, edge_threshold=8.0,
        warmup_runs=3, timed_runs=10,
    )
    checkpoint_csv = METRICS_ROOT / f'{dataset}_x2_nedi_final.csv'
    pairs = pair_image_paths(validation.hr_directory, validation.lr_directory)
    if checkpoint_csv.exists():
        with checkpoint_csv.open(newline='', encoding='utf-8') as file:
            dataset_records = list(csv.DictReader(file))
        completed_images = {record['image'] for record in dataset_records}
        print(f'Resuming {dataset} x2: {len(completed_images)}/{len(pairs)} images already complete.')
    else:
        dataset_records = []
        completed_images = set()

    for index, (hr_path, lr_path) in enumerate(pairs, start=1):
        if hr_path.name in completed_images:
            continue
        record = evaluate_nedi_image(hr_path, lr_path, config)
        dataset_records.append(record)
        write_results_csv(dataset_records, checkpoint_csv, overwrite=True)
        print(
            f'{dataset} x2: {index}/{len(pairs)} — {hr_path.name} — '
            f'PSNR-Y={record["psnr_y"]:.4f}, time={record["latency_mean_ms"] / 1000:.2f}s'
        )

    all_records.extend(dataset_records)
    print(f'Completed {dataset} x2: {checkpoint_csv}')

print(f'Completed {len(all_records)} NEDI x2 image evaluations.')


In [ ]:
combined_csv = METRICS_ROOT / 'nedi_x2_all_images_final.csv'
summary_csv = METRICS_ROOT / 'nedi_x2_summary_final.csv'
summary_records = summarize_results(all_records)
write_results_csv(all_records, combined_csv)
write_results_csv(summary_records, summary_csv)

for row in summary_records:
    print(
        f"{row['dataset']} x2: images={row['image_count']}, "
        f"PSNR-Y={row['psnr_y']:.4f}, SSIM-Y={row['ssim_y']:.4f}, "
        f"PSNR-RGB={row['psnr_rgb']:.4f}, SSIM-RGB={row['ssim_rgb']:.4f}, "
        f"latency={row['latency_mean_ms']:.2f} ms"
    )

print(f'Combined results: {combined_csv}')
print(f'Summary: {summary_csv}')
